In [2]:
import sys
import subprocess
import mcp.client.stdio
import mcp.os.win32.utilities
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

# Fix Jupyter on Windows fileno issue with MCP stdio
mcp.client.stdio.stdio_client.__wrapped__.__defaults__ = (sys.__stderr__ or subprocess.DEVNULL,)
mcp.os.win32.utilities._create_windows_fallback_process.__defaults__ = (None, sys.__stderr__ or subprocess.DEVNULL, None)

load_dotenv()

True

In [3]:
client = MultiServerMCPClient(
    {
        "time":{
            "transport":"stdio",
            "command":"uvx",
            "args":[
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

In [4]:
tools = await client.get_tools()

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=tools
)


In [8]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is the time now?")]}
)

In [13]:
response['messages'][-1].content[0]['text']

'The current time in the local timezone (America/New_York) is **9:13 AM** on **Friday, September 4, 2026** (EDT).'